<a href="https://colab.research.google.com/github/AnnaMilagres/segmentacao-clientes/blob/main/segmentacao_clientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentação de Clientes de uma Farmácia por Comportamento de Consumo

## Mineração de Dados - Atividade Prática

Instituto Federal de Educação, Ciência e Tecnologia da Paraíba - IFPB

Curso: Engenharia de Computação

Disciplina: Mineração de Dados

Professor: Marcelo José Siqueira Coutinho de Almeida

Aluno(s): Anna Milagres Albino de Oliveira

## 1. Demanda

Uma farmácia deseja compreender melhor os diferentes perfis de seus clientes a partir de seus hábitos de consumo.

### Quais problemas que a organização deseja resolver?

Identificar grupos de clientes que apresentam comportamentos de consumo semelhantes a partir de seu histórico de compras.

### Quem utilizará os resultados?

Os resultados serão utilizados pelos responsáveis pela gestão da farmácia para compreender melhor os perfis de seus clientes

### Qual a utilidade prática da solução?

Esse conhecimento sobre o consumo de cada grupo pode auxiliar na identificação de padrões e na elaboração de estratégias específicas para cada perfil de consumidor, por exemplo na organização da área exposta aos clientes.

## 2. Formulação do problema de mineração

**Problema**: Segmentação de clientes com base em seus comportamentos de consumo.

**Tarefa de mineração**: Agrupamento/Clustering.

**Variável-alvo**: Não há variável-alvo, pois trata-se de uma tarefa não supervisionada.

**Unidade de observação**: Cliente.

**Objetivo**: Identificar grupos de clientes com características de consumo semelhantes utilizando informações provenientes de seu histórico de compras.

## 3. Dados Necessários

| Variável | Descrição | Função |
|---|---|---|
| ID do cliente | Identificador anônimo do cliente | Identificação |
| Número de compras | Quantidade de compras realizadas | Atributo |
| Valor total gasto | Soma dos valores das compras | Atributo |
| Valor médio por compra | Gasto médio por compra | Atributo |
| Frequência de compra | Frequência com que o cliente realiza compras | Atributo |

## 4. Fonte escolhida e justificativa

### Documentação da Fonte:

**Nome da fonte**: Sistema de gestão de vendas e PDV de farmácia privada (dados reais).

**Nome do dataset**: vendas_anonimizadas.csv

**Origem dos dados**: Extração direta do banco de dados do estabelecimento, refletindo as transações reais do último ano.

**Descrição da base**: O dataset contém o histórico detalhado de cupons fiscais e vendas, incluindo a identificação do cliente, produtos adquiridos, quantidades, preço unitário e valor total da transação.

**Condições de uso**: Uso acadêmico restrito. Os dados passaram por um pré-processamento rigoroso de limpeza e anonimização (aplicação de hash SHA-256 em CPFs/CNPJs e remoção de nomes explícitos) para garantir a privacidade dos clientes e a conformidade com a LGPD.

#### Justificativa da Escolha:

A base foi considerada adequada para o problema proposto pois atende diretamente aos critérios de compatibilidade com a demanda de segmentação de clientes. Diferente de datasets genéricos, esta base fornece atributos cruciais e reais sobre o comportamento de compra (frequência, valor gasto, tipos de produtos adquiridos). A riqueza dessas variáveis permite a aplicação robusta de algoritmos de agrupamento (clustering) para identificar nichos de consumidores de forma muito mais fidedigna à realidade do mercado varejista farmacêutico.


## 5. Aquisição dos dados

Nesta etapa, realizamos a importação da base de dados proveniente do sistema da farmácia. Como o arquivo original passou por um script de pré-processamento para limpeza e anonimização (detalhado no anexo deste notebook), faremos a leitura do arquivo final `vendas_anonimizadas.csv` utilizando a biblioteca `pandas`.

In [13]:
import pandas as pd

caminho_arquivo = "../data/vendas_anonimizadas.csv"
df = pd.read_csv(caminho_arquivo, sep=";")
df.head()


,venda_id,data,hora,cliente_id,produto_id,produto,quantidade,preco_unitario,valor_total
0,335659,21/08/2025,07:54:28,NaN,9952,DICLO SODICO 50MG C/20-GEOLAB,1.0,NaN,NaN
1,335660,21/08/2025,08:03:32,C7dfc5e6365,13600,LACRIFILM COLIRIO 10ML,1.0,NaN,NaN
2,335660,21/08/2025,08:03:32,C7dfc5e6365,23114,"XAFAC 2,5MG 60CPR SN",1.0,NaN,NaN
3,335660,21/08/2025,08:03:32,C7dfc5e6365,12535,SINVASTATINA 20MG C/30C-GN SAN,1.0,NaN,NaN
4,335660,21/08/2025,08:03:32,C7dfc5e6365,5567,LOSARTANA POT 50MG C/30-GN,1.0,NaN,NaN


## 6. Inspeção Inicial

Nesta etapa, utilizamos comandos nativos do Pandas para verificar as dimensões do dataset, os tipos de variáveis, a presença de valores ausentes e a ocorrência de registros duplicados, a fim de avaliar a integridade inicial da base de dados.

In [ ]:
# Quantidade de linhas e colunas
print("Dimensões do dataset (linhas, colunas):", df.shape)

# Nome das colunas principais
print("\nColunas presentes:", df.columns.tolist())

# Informações gerais e tipos de dados
print("\n--- Informações do DataFrame ---")
df.info()

# Valores ausentes
print("\n--- Valores Ausentes ---")
print(df.isnull().sum())

# Registros duplicados
print("\n--- Linhas Duplicadas ---")
print("Total de duplicadas:", df.duplicated().sum())

# Resumo estatístico
print("\n--- Resumo Estatístico ---")
display(df.describe())

Com base nos comandos executados, a inspeção inicial revelou as seguintes características da base de dados:

**Quantas linhas existem?** 
O dataset possui 48.155 registros de itens vendidos.  Quantas colunas existem? Existem 9 colunas estruturando os dados.  

**Quais são as principais variáveis?**
Para a tarefa de agrupamento, as variáveis mais relevantes são cliente_id, quantidade, preco_unitario e valor_total.  

**Quais são os tipos de dados?**
A base é composta por variáveis do tipo int64 (inteiros, como os IDs de venda e produto), float64 (decimais, como preço e quantidade) e str (texto, para datas, horários, IDs com letras e nomes de produtos).  

**Existem valores ausentes?** 
Sim. A coluna cliente_id não possui identificação em 41.164 registros, o que é um comportamento esperado no varejo farmacêutico para consumidores casuais. As colunas de preço apresentam apenas 19 valores ausentes.  

**Existem registros duplicados?** 
Foram identificadas 732 linhas duplicadas no dataset.  

**Existem valores aparentemente inconsistentes?** 
Após a correção no script de extração, não há mais valores negativos na coluna de total. Apenas a quantidade máxima de 300 unidades de um único produto em uma venda chama a atenção como um possível outlier (compra de atacado ou institucional). 

## 7. Avaliação da qualidade e adequação

**Classificação**: Adequada.  

**Justificativa**: A base reflete transações reais e, após o tratamento estrutural das colunas, demonstrou alta consistência matemática. Apesar da grande quantidade de clientes não identificados (comum no setor), o subconjunto de clientes fidelizados (161 consumidores, responsáveis por 6.991 itens comprados) possui histórico rico, com número de compras variando de 1 a 140. Esses dados fornecem volume e variação perfeitamente adequados para aplicar algoritmos de agrupamento e encontrar padrões de consumo.  